# Module `solver_input.py`

Ce notebook presente le contrat d'entree partage par **tous** les algorithmes de resolution du projet.

Principe : un solveur ne doit jamais lire directement un `GraphInstance` ou un `DynamicGraphSnapshot`. Il consomme uniquement un `SolverInput`, ce qui garantit l'interchangeabilite des sources de donnees.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.dynamic_network import DynamicNetworkSimulator
from cesipath.graph_generator import GraphGenerator
from cesipath.models import GraphGenerationConfig
from cesipath.solver_input import SolverInput, build_dynamic_solver_input, build_static_solver_input

## 1. Les champs du contrat

`SolverInput` est une dataclass frozen. Ses champs :

| Champ | Type | Role |
|---|---|---|
| `cost_matrix` | `list[list[float]]` | Matrice complete deja metrique |
| `depot` | `int` | Index du sommet depot |
| `demands` | `dict[int, int]` | Demande par sommet, `0` sur le depot |
| `vehicle_capacity` | `int` | Capacite max par tournee |
| `shortest_paths` | `dict[(int, int), list[int]]` | Vraies sequences de sommets derriere chaque arete virtuelle |
| `source` | `str` | `"static"` ou `"dynamic"` |
| `dynamic_step` | `int | None` | Numero de tour si dynamique, sinon `None` |

Le champ `shortest_paths` est crucial pour `execute_dynamic` : quand un vehicule doit effectivement traverser une arete virtuelle, on reconstruit le trajet reel pour comptabiliser correctement les couts dynamiques.

## 2. Construction statique

`build_static_solver_input(instance)` extrait les donnees de l'instance statique :

- la matrice `completed_costs` (passage Dijkstra sur les aretes non FORBIDDEN) ;
- les `completed_paths` associes ;
- `source="static"`, `dynamic_step=None`.

Le solveur n'a plus a savoir qu'il existe un graphe de base, des aretes surchargees ou des coordonnees.

In [ ]:
config = GraphGenerationConfig(node_count=8, seed=42)
instance = GraphGenerator(config).generate()

static_input = build_static_solver_input(instance)
print("Type         :", type(static_input).__name__)
print("source       :", static_input.source)
print("depot        :", static_input.depot)
print("capacity     :", static_input.vehicle_capacity)
print("demandes     :", static_input.demands)
print("nb sommets   :", len(static_input.cost_matrix))
print("chemin 0->3  :", static_input.shortest_paths.get((0, 3)))

## 3. Construction dynamique

`build_dynamic_solver_input(instance, snapshot)` produit la meme structure mais :

- la matrice et les chemins viennent du **snapshot** courant ;
- `source="dynamic"` ;
- `dynamic_step` est renseigne, ce qui permet a un solveur de savoir a quel tour il est sans connaitre le simulateur.

In [ ]:
simulator = DynamicNetworkSimulator(instance, seed=42)
snapshot = simulator.advance(simulator.initialize_snapshot())

dynamic_input = build_dynamic_solver_input(instance, snapshot)
print("source       :", dynamic_input.source)
print("dynamic_step :", dynamic_input.dynamic_step)
print("1ere ligne   :", dynamic_input.cost_matrix[0])

## 4. Usage dans les algorithmes

Tous les algorithmes de `cesipath.algorithms` suivent la meme signature :

```python
solution = algo(solver_input, **hyperparams)
```

ou `algo` peut etre `grasp`, `simulated_annealing`, `tabu_search`, `genetic_algorithm`. Le solveur consulte uniquement :

- `cost_matrix` pour evaluer les couts ;
- `demands` et `vehicle_capacity` pour verifier la faisabilite ;
- `depot` pour savoir ou la tournee commence/finit.

Le champ `source` permet a un benchmark dynamique d'adapter son affichage (ex. "planifie vs realise").

In [ ]:
from cesipath.algorithms import grasp

solution = grasp(static_input, max_iterations=5, rcl_alpha=0.2, seed=42)
print("Nb tournees :", solution.route_count)
print("Cout total  :", round(solution.total_cost, 2))
print("Tournees    :", solution.routes)

## 5. Pourquoi un type frozen ?

`SolverInput` est immuable. Un solveur ne peut pas modifier la matrice ou la demande en cours de route. Consequence :

- un benchmark peut lancer plusieurs algorithmes sur le meme `SolverInput` sans craindre la contamination croisee ;
- les tests de reproductibilite sont plus simples (pas d'effet de bord) ;
- si un solveur veut une matrice modifiee (ex. penalisation), il doit explicitement construire un nouvel objet.